# 📊 Módulo 2 (visual) — Treinar um mini-GPT do zero com gráficos ao vivo

Este notebook acompanha o **Módulo 2** do guia (`docs/02_GUIA_DE_APRENDIZADO.md`).
Aqui você treina o mesmo mini-GPT do script `scripts/02_train_tiny_gpt.py`, mas de
forma **interativa e visual**: vamos plotar a curva de *loss* e a *perplexity* em
matplotlib e gerar texto ao final.

> Antes de rodar: abra este notebook com `make lab` (JupyterLab) a partir da raiz do
> projeto, com o `.venv` ativo.

**O que você vai observar:** a loss (erro) caindo passo a passo — é o modelo
aprendendo a prever o próximo caractere.

## 1. Setup — imports e checagem da GPU

In [ ]:
import os
import math
import importlib.util

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
import matplotlib.pyplot as plt

# Raiz do projeto (um nível acima de notebooks/)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
print("Projeto:", ROOT)
print("Device MLX:", mx.default_device())  # deve mostrar Device(gpu, 0)

## 2. Reaproveitar o modelo do script `02`

Em vez de copiar o código do Transformer, carregamos as classes já escritas em
`scripts/02_train_tiny_gpt.py` (o nome começa com número, então usamos `importlib`).
Assim você vê que notebook e script compartilham o mesmo modelo.

In [ ]:
spec = importlib.util.spec_from_file_location(
    "tiny_gpt", os.path.join(ROOT, "scripts", "02_train_tiny_gpt.py")
)
tiny_gpt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tiny_gpt)
MiniGPT = tiny_gpt.MiniGPT
print("MiniGPT carregado:", MiniGPT)

## 3. Carregar o corpus e tokenizar (char-level)

Se ainda não gerou os dados, rode `make data` no terminal (ou
`python scripts/01_prepare_data.py`). Cada caractere único vira um número (id).

In [ ]:
corpus_path = os.path.join(ROOT, "data", "processed", "corpus.txt")
assert os.path.exists(corpus_path), "Rode 'make data' antes!"
text = open(corpus_path, encoding="utf-8").read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

data = mx.array([stoi[c] for c in text], dtype=mx.int32)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

print(f"Vocabulário: {vocab_size} caracteres")
print(f"Total de tokens: {len(data):,}")
print(f"Amostra do texto: {text[:80]!r}")

## 4. Hiperparâmetros e modelo

Mexa nestes números e re-execute para ver o efeito no treino.

In [ ]:
STEPS      = 800    # quantos passos de treino
BATCH      = 32     # exemplos por passo
BLOCK_SIZE = 64     # contexto (caracteres que o modelo vê)
N_EMBD     = 128    # dimensão dos embeddings
N_HEAD     = 4      # cabeças de atenção
N_LAYER    = 4      # blocos Transformer
LR         = 3e-3   # learning rate

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = mx.random.randint(0, len(d) - BLOCK_SIZE, (BATCH,))
    x = mx.stack([d[i:i + BLOCK_SIZE] for i in ix.tolist()])
    y = mx.stack([d[i + 1:i + BLOCK_SIZE + 1] for i in ix.tolist()])
    return x, y

from mlx.utils import tree_flatten

model = MiniGPT(vocab_size, N_EMBD, N_HEAD, N_LAYER, BLOCK_SIZE)
mx.eval(model.parameters())

total = sum(v.size for _, v in tree_flatten(model.parameters()))
print(f"Modelo criado com {total:,} parâmetros "
      f"({N_LAYER} camadas, {N_EMBD} dims, {N_HEAD} cabeças).")

## 5. Loop de treino coletando as métricas

Guardamos `train_loss` e `val_loss` em listas para plotar depois. Isso é
exatamente o que o MLflow faz por baixo — aqui fazemos à mão para enxergar.

In [ ]:
def loss_fn(model, x, y):
    logits = model(x)
    B, T, C = logits.shape
    return nn.losses.cross_entropy(logits.reshape(B * T, C), y.reshape(B * T)).mean()

optimizer = optim.AdamW(learning_rate=LR)
loss_and_grad = nn.value_and_grad(model, loss_fn)

steps_hist, train_hist, val_hist = [], [], []

for step in range(STEPS):
    x, y = get_batch("train")
    loss, grads = loss_and_grad(model, x, y)
    optimizer.update(model, grads)
    mx.eval(model.parameters(), optimizer.state)

    if step % 25 == 0 or step == STEPS - 1:
        vx, vy = get_batch("val")
        vloss = loss_fn(model, vx, vy)
        mx.eval(vloss)
        steps_hist.append(step)
        train_hist.append(float(loss))
        val_hist.append(float(vloss))
        print(f"step {step:4d} | train {float(loss):.3f} | "
              f"val {float(vloss):.3f} | ppl {math.exp(float(vloss)):.1f}")

print("\nTreino concluído!")

## 6. 📈 Gráficos: curva de loss e de perplexity

A loss deve cair e depois estabilizar. Se a `val_loss` começar a **subir**
enquanto a `train_loss` cai, isso é sinal de **overfitting**.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(steps_hist, train_hist, label="train", marker="o", ms=3)
ax1.plot(steps_hist, val_hist, label="val", marker="s", ms=3)
ax1.set_title("Loss (cross-entropy)")
ax1.set_xlabel("passo")
ax1.set_ylabel("loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ppl = [math.exp(v) for v in val_hist]
ax2.plot(steps_hist, ppl, color="crimson", marker="d", ms=3)
ax2.set_title("Perplexity (val) — menor é melhor")
ax2.set_xlabel("passo")
ax2.set_ylabel("perplexity")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Gerar texto com o modelo treinado

Damos um caractere inicial e o modelo continua, prevendo um caractere por vez.
Com poucos passos e corpus de brinquedo, o texto será imperfeito — o objetivo
é ver o **mecanismo** funcionando.

In [ ]:
context = mx.array([[stoi.get("M", 0)]])
out = model.generate(context, max_new_tokens=300)[0].tolist()
print("".join(itos[i] for i in out))

## 8. 🧪 Exercícios

Volte na célula 4, mude e re-execute as células 4→7:

1. **Mais treino:** `STEPS = 3000`. A curva continua caindo? O texto melhora?
2. **Modelo maior:** `N_LAYER = 6`, `N_EMBD = 256`. Compare a perplexity final.
3. **Learning rate:** teste `LR = 1e-2` e `LR = 1e-3`. O que muda na curva?
4. **Overfitting:** reduza o corpus (edite `01_prepare_data.py` para repetir
   menos vezes) e veja se a `val_loss` volta a subir.

📝 Anote suas observações no `docs/03_DIARIO.md`.